# Open-question evaluation — 30 fine-tuned models

Run all cells on Kaggle with a T4 GPU, Internet, and HF_TOKEN if needed.
Reads CT/test/open.csv with case-insensitive folder matching (Test/test).
Both training variants receive the SAME image/question prompt. No test CoT
column is required or read. Models use Arup330/{Dataset}_open_{CoT|noCoT}_{Model}_lora.

The final section prints three 10-model tables and one combined 30-model table:
BLEU, ROUGE-1, ROUGE-2, ROUGE-L, BERTScore F1/P/R. No classification accuracy.
Model names reflect your actual repositories, including MedGemma-4B.

BLEU is corpus BLEU-4 with NLTK method1 smoothing and whitespace tokenization;
ROUGE is mean per-example F1 without stemming; BERTScore uses roberta-large,
no IDF and no baseline rescaling. Text is lowercased and whitespace normalized.
The screenshot defines the layout, not the precise metric implementation; the
chosen settings are exported for reproducibility. BLEU-4 can be low for short
correct answers because higher-order n-grams are absent.

Predictions print per question and save after each model. BERTScore runs once
all VLM inference finishes, so the VLM and scorer do not share GPU memory.
All 30 results are required. Smoke-test reports (MAX_ROWS_PER_DATASET=5) are
clearly labelled. Set it to None for all test rows. Scores are decimal values.

The install cell does not force upgrades of an already working Unsloth setup.
Metric-library imports are checked before inference. Numeric reference answers
are preserved as CSV text. The final section validates saved predictions and
metric settings before displaying results, preventing stale summaries.


In [5]:
%pip install -q unsloth bert-score rouge-score nltk pillow


Note: you may need to restart the kernel to use updated packages.


In [6]:
import os, re, gc, json, glob, time, shutil, hashlib
import unicodedata
from collections import OrderedDict
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
from unsloth import FastVisionModel
import torch
import pandas as pd
from PIL import Image
from huggingface_hub import login, HfApi

# ---------------------------------------------------------------------------
# Hugging Face token. Needed if your LoRA repos are private or a base model is
# gated (e.g. google/medgemma-4b-it).
# In Kaggle: Add-ons -> Secrets -> add HF_TOKEN -> attach it to this notebook.
# ---------------------------------------------------------------------------
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF_Token")
try:
    from kaggle_secrets import UserSecretsClient
    for secret_name in ("HF_TOKEN", "HF_Token"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(secret_name) or HF_TOKEN
            if HF_TOKEN:
                break
        except Exception:
            continue
except Exception:
    pass

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
else:
    print("No HF_TOKEN found -- loading a private or gated dependency may fail.")

Logged in to Hugging Face.


## 1. Configuration

In [7]:
# ---------------------------------------------------------------------------
# 1. Configuration -- everything you may need to change is in this cell
# ---------------------------------------------------------------------------
HF_USER = "Arup330"

# Folder that contains the dataset folders (Abdomen/, Neck/, BrainFace/ ...)
DATA_ROOT = "/kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages"

# Name used in the Hugging Face repo names -> folder name inside DATA_ROOT.
# The test set is read from <DATA_ROOT>/<folder>/<modality>/test/open.csv
DATASETS = {
    "BrainFace": "Brain_Face",
    "Neck":      "Neck",
    "Abdomen":   "Abdomen",
}

# Model suffixes used by your Hugging Face repositories.
MODELS = [
    "Lingshu-7B",
    "MedGemma-4B",
    "Qwen2.5-VL-7B",
    "Qwen3-VL-8B",
    "Llama-11B",
]

# (label in the summary table, tag in the repo name)
CONDITIONS = [("With", "CoT"), ("Without", "noCoT")]

# Override only repositories that use a different naming pattern.
# Key: (dataset name, summary condition, model suffix).
MODEL_REPO_OVERRIDES = {}
# Example: ("BrainFace", "With", "Llama-11B"): "Arup330/EXACT_REPO_NAME"

def model_repo(dataset: str, tag: str, model_name: str) -> str:
    condition = next(label for label, value in CONDITIONS if value == tag)
    return MODEL_REPO_OVERRIDES.get(
        (dataset, condition, model_name),
        f"{HF_USER}/{dataset}_open_{tag}_{model_name}_lora",
    )

MODALITY = "CT"
BERTSCORE_MODEL = "roberta-large"
BERTSCORE_BATCH_SIZE = 16

# All 30 are required for the final report. Subsets allow work across sessions.
RUN_DATASETS = list(DATASETS)  # e.g. ["BrainFace"] for one session
RUN_MODELS = list(MODELS)     # e.g. ["Llama-11B"] to resume one family
# Optional previous Kaggle output folders containing raw_hf_models:
RESUME_RAW_DIRS = []  # e.g. ["/kaggle/input/previous-results/open_eval_results/raw_hf_models"]
assert set(RUN_DATASETS) <= set(DATASETS)
assert set(RUN_MODELS) <= set(MODELS)

MAX_ROWS_PER_DATASET = None   # e.g. 5 for a quick smoke test of all 30 runs
MAX_IMAGE_SIDE = None         # e.g. 1024 to downscale very large images (faster, less VRAM)
MAX_NEW_TOKENS = 32
IMAGE_CACHE_MB = 256  # Decoded RGB images in CPU RAM, not GPU memory.
if MAX_ROWS_PER_DATASET is not None and (
    type(MAX_ROWS_PER_DATASET) is not int or MAX_ROWS_PER_DATASET <= 0
):
    raise ValueError("MAX_ROWS_PER_DATASET must be None or a positive integer.")
if MAX_NEW_TOKENS <= 0 or BERTSCORE_BATCH_SIZE <= 0:
    raise ValueError("Token limit and metric batch size must be positive.")

# Every finished run is saved here (shown in the notebook's Output panel).
# Re-running the notebook in the same session continues where it stopped.
RESULTS_DIR = "/kaggle/working/open_eval_results"
RAW_DIR = os.path.join(RESULTS_DIR, "raw_hf_models")
os.makedirs(RAW_DIR, exist_ok=True)
for dataset in RUN_DATASETS:
    print(f"\n{dataset}: 10 open-question repositories")
    for model_name in RUN_MODELS:
        for condition, tag in CONDITIONS:
            print(f"  {condition}: {model_repo(dataset, tag, model_name)}")



BrainFace: 10 open-question repositories
  With: Arup330/BrainFace_open_CoT_Lingshu-7B_lora
  Without: Arup330/BrainFace_open_noCoT_Lingshu-7B_lora
  With: Arup330/BrainFace_open_CoT_MedGemma-4B_lora
  Without: Arup330/BrainFace_open_noCoT_MedGemma-4B_lora
  With: Arup330/BrainFace_open_CoT_Qwen2.5-VL-7B_lora
  Without: Arup330/BrainFace_open_noCoT_Qwen2.5-VL-7B_lora
  With: Arup330/BrainFace_open_CoT_Qwen3-VL-8B_lora
  Without: Arup330/BrainFace_open_noCoT_Qwen3-VL-8B_lora
  With: Arup330/BrainFace_open_CoT_Llama-11B_lora
  Without: Arup330/BrainFace_open_noCoT_Llama-11B_lora

Neck: 10 open-question repositories
  With: Arup330/Neck_open_CoT_Lingshu-7B_lora
  Without: Arup330/Neck_open_noCoT_Lingshu-7B_lora
  With: Arup330/Neck_open_CoT_MedGemma-4B_lora
  Without: Arup330/Neck_open_noCoT_MedGemma-4B_lora
  With: Arup330/Neck_open_CoT_Qwen2.5-VL-7B_lora
  Without: Arup330/Neck_open_noCoT_Qwen2.5-VL-7B_lora
  With: Arup330/Neck_open_CoT_Qwen3-VL-8B_lora
  Without: Arup330/Neck_open_noC

## 2. Open-question test data


In [8]:
# ---------------------------------------------------------------------------
# 2. Load every open-question TEST row; answers are free text
#    Reads <DATA_ROOT>/<folder>/<modality>/test/open.csv directly -- no zip.
# ---------------------------------------------------------------------------
def normalize_answer(text) -> str:
    # Do not truncate words, remove negations, or map synonyms using gold labels.
    return " ".join(unicodedata.normalize("NFKC", str(text)).lower().split())

def find_test_csv(folder):
    # Match capitalization only; never silently select another anatomical cohort.
    def child(parent, name):
        if not os.path.isdir(parent):
            raise FileNotFoundError(f"Directory not found: {parent}")
        matches = [os.path.join(parent, x) for x in os.listdir(parent) if x.casefold() == name.casefold()]
        if len(matches) != 1:
            raise FileNotFoundError(f"Expected one {name!r} under {parent}; found {matches}")
        return matches[0]
    path = child(DATA_ROOT, folder)
    path = child(path, MODALITY)
    path = child(path, "test")
    return child(path, "open.csv")

def resolve_image_path(split_dir: str, img_name: str):
    img_name = str(img_name)
    for c in [os.path.join(split_dir, img_name),
              os.path.join(split_dir, os.path.basename(img_name)),
              os.path.join(split_dir, img_name.replace("/", "_"))]:
        if os.path.exists(c):
            return c
    return None

# Bound CPU memory while reusing repeated images across questions/models.
IMAGE_CACHE = OrderedDict()
IMAGE_CACHE_BYTES = 0

def load_image(path: str) -> Image.Image:
    global IMAGE_CACHE_BYTES
    key = (os.path.abspath(path), MAX_IMAGE_SIDE)
    if key in IMAGE_CACHE:
        IMAGE_CACHE.move_to_end(key)
        return IMAGE_CACHE[key].copy()
    with Image.open(path) as source:
        img = source.convert("RGB")
    if MAX_IMAGE_SIDE and max(img.size) > MAX_IMAGE_SIDE:
        img.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.LANCZOS)
    size_bytes = img.width * img.height * 3
    budget = int(IMAGE_CACHE_MB * 1024 * 1024)
    if size_bytes <= budget:
        while IMAGE_CACHE and IMAGE_CACHE_BYTES + size_bytes > budget:
            _, old = IMAGE_CACHE.popitem(last=False)
            IMAGE_CACHE_BYTES -= old.width * old.height * 3
            old.close()
        IMAGE_CACHE[key] = img
        IMAGE_CACHE_BYTES += size_bytes
        return img.copy()  # Caller can close/mutate this safely.
    return img

def load_open_test(name: str, folder: str) -> pd.DataFrame:
    csv_path = find_test_csv(folder)
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    required = {"question", "answer"}
    if not required <= set(df.columns):
        raise ValueError(f"{csv_path}: missing columns {required - set(df.columns)}")
    img_col = next((c for c in ("image_file", "img_name") if c in df.columns), None)
    if img_col is None:
        raise ValueError(f"{csv_path}: requires image_file or img_name")
    if df.empty:
        raise ValueError(f"{name}: empty open-question test set")
    if "answer_type" in df and not df["answer_type"].astype(str).str.strip().str.upper().eq("OPEN").all():
        raise ValueError(f"{csv_path}: non-OPEN rows found; check the input file")
    for column in ("question", "answer"):
        if df[column].astype(str).str.strip().eq("").any():
            raise ValueError(f"{name}: empty {column} in the test set")
    df["question"] = df["question"].astype(str)
    df["gold"] = df["answer"].astype(str)
    df["img_path"] = [resolve_image_path(os.path.dirname(csv_path), image) for image in df[img_col]]
    if df["img_path"].isna().any():
        raise FileNotFoundError(f"{name}: unresolved images {df.loc[df['img_path'].isna(), img_col].tolist()[:5]}")
    columns = ["question", "gold", "img_path"]
    if MAX_ROWS_PER_DATASET is not None:
        df = df.head(MAX_ROWS_PER_DATASET)
    df = df[columns].reset_index(drop=True)
    print(f"{name}: {len(df)} open questions, {df.img_path.nunique()} unique images | {csv_path}")
    return df

TEST = {name: load_open_test(name, folder) for name, folder in DATASETS.items()}
print(f"Total model predictions: {10 * sum(map(len, TEST.values()))}")


BrainFace: 17 open questions, 2 unique images | /kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Brain_Face/CT/test/open.csv
Neck: 19 open questions, 3 unique images | /kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Neck/CT/test/open.csv
Abdomen: 160 open questions, 22 unique images | /kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Abdomen/CT/test/open.csv
Total model predictions: 1960


## 3. Prompts

In [9]:
def systemPrompt(question: str) -> str:
    return f"""Context:
You are a medical visual question answering specialist.

Objective:
Answer the question directly using the visual evidence in the provided medical image.

Question: {question}

Instructions:
1. Examine the image and identify the structures or findings relevant to the question.
2. Interpret the question carefully, including negation and anatomical relationships.
3. Give the most specific answer supported by the image.
4. Do not invent findings or add unsupported details.

Output Requirements:
- Return only the concise final answer, preferably in 1 to 4 words.
- Use additional words only when necessary for a complete answer.
- Do not include explanations, introductory phrases, or unnecessary punctuation.
"""


## 4. Model helpers

In [10]:
# Load your fine-tuned Hugging Face repository using Unsloth.
def load_model(repo: str):
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a CUDA GPU before running model evaluation.")
    model, tokenizer = FastVisionModel.from_pretrained(
        repo,
        load_in_4bit=True,
        dtype=torch.float16,
        device_map={"": "cuda:0"},
        use_gradient_checkpointing="unsloth",
        token=HF_TOKEN,
    )
    FastVisionModel.for_inference(model)
    return model, tokenizer

@torch.inference_mode()
def generate_answer(model, tokenizer, image, prompt_text: str) -> str:
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt_text},
        ],
    }]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(
        images=image, text=input_text,
        add_special_tokens=False, return_tensors="pt",
    ).to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True)
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def run_condition(model, tokenizer, df: pd.DataFrame, tag: str):
    results, t0 = [], time.time()
    for i, row in enumerate(df.itertuples(index=False)):
        prompt = systemPrompt(row.question)
        image = load_image(row.img_path)
        try:
            raw = generate_answer(model, tokenizer, image, prompt)
        except Exception as e:
            raise RuntimeError(f"{tag}: test row {i} failed: {e}") from e
        finally:
            image.close()
        results.append({
            "row": i, "image": os.path.basename(row.img_path), "question": row.question,
            "gold": row.gold, "pred": normalize_answer(raw), "raw": raw, "error": None,
        })
        print(f"    [{i}/{len(df)}] gold={row.gold!r} pred={normalize_answer(raw)!r} raw={raw!r}", flush=True)
    return results


## 5. Run all models

In [11]:
# 5 models x 3 datasets x 2 training variants = 30 fine-tuned models.
# Every run loads its own Arup330 repository, evaluates the same prompt,
# and saves predictions for the existing summary section.
def result_path(dataset, model_name, condition):
    return os.path.join(RAW_DIR, f"{dataset}__{model_name}__{condition}.json")

IMAGE_HASHES = {}

def image_hash(path):
    if path not in IMAGE_HASHES:
        digest = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                digest.update(chunk)
        IMAGE_HASHES[path] = digest.hexdigest()
    return IMAGE_HASHES[path]

def run_signature(dataset, model_name, condition):
    # Content hashes allow results to resume after Kaggle remounts saved files.
    rows = [{"question": str(row.question), "gold": row.gold,
             "image_sha256": image_hash(row.img_path)}
            for row in TEST[dataset].itertuples(index=False)]
    payload = {
        "repo": model_repo(dataset, dict(CONDITIONS)[condition], model_name),
        "rows": rows,
        "tokens": MAX_NEW_TOKENS,
        "image_side": MAX_IMAGE_SIDE,
        "prompts": [systemPrompt(row.question) for row in TEST[dataset].itertuples(index=False)],
        "version": "open-v2-shared-prompt",
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def valid_result_file(path, dataset, model_name, condition):
    try:
        with open(path) as f:
            rows = json.load(f)
        with open(path + ".meta.json") as f:
            meta = json.load(f)
        return (len(rows) == len(TEST[dataset])
                and not any(r.get("error") for r in rows)
                and meta["signature"] == run_signature(dataset, model_name, condition))
    except (OSError, ValueError, KeyError, TypeError):
        return False

def is_done(dataset, model_name, condition):
    return valid_result_file(result_path(dataset, model_name, condition),
                             dataset, model_name, condition)

def restore_results():
    for dataset in DATASETS:
        for model_name in MODELS:
            for condition, _ in CONDITIONS:
                if is_done(dataset, model_name, condition):
                    continue
                target = result_path(dataset, model_name, condition)
                for directory in RESUME_RAW_DIRS:
                    source = os.path.join(directory, os.path.basename(target))
                    if valid_result_file(source, dataset, model_name, condition):
                        shutil.copy2(source, target)
                        shutil.copy2(source + ".meta.json", target + ".meta.json")
                        print(f"[RESTORED] {dataset} | {model_name} | {condition}")
                        break

def save_results(dataset, model_name, condition, results):
    p = result_path(dataset, model_name, condition)
    with open(p + ".tmp", "w") as f:
        json.dump(results, f, indent=2)
    os.replace(p + ".tmp", p)
    with open(p + ".meta.json", "w") as f:
        json.dump({"signature": run_signature(dataset, model_name, condition),
                   "model_repo": model_repo(dataset, dict(CONDITIONS)[condition], model_name)}, f, indent=2)


# Catch missing metric dependencies before spending time on 30 model runs.
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import BERTScorer

# Check all repositories with the user's token before downloading any weights.
api = HfApi(token=HF_TOKEN)
repo_errors = []
for dataset in DATASETS:
    for model_name in MODELS:
        for condition, tag in CONDITIONS:
            repo = model_repo(dataset, tag, model_name)
            try:
                api.model_info(repo)
                print(f"[ACCESS OK] {repo}")
            except Exception as e:
                repo_errors.append(f"{repo}: {type(e).__name__}: {e}")
if repo_errors:
    raise RuntimeError("Fix repository names/access before all 30 evaluations:\n" + "\n".join(repo_errors))

restore_results()
run_start = time.time()
RUN_TIMINGS = []

for model_name in RUN_MODELS:
    for dataset in RUN_DATASETS:
        for condition, tag in CONDITIONS:
            repo = model_repo(dataset, tag, model_name)
            if is_done(dataset, model_name, condition):
                print(f"[DONE] {repo}: reusing verified saved results")
                continue
            model = tokenizer = None
            try:
                print(f"\n--- Loading {repo} ---")
                load_start = time.time()
                model, tokenizer = load_model(repo)
                load_seconds = time.time() - load_start
                inference_start = time.time()
                results = run_condition(
                    model, tokenizer, TEST[dataset],
                    tag=f"{dataset} | {model_name} | {condition} CoT",
                )
                inference_seconds = time.time() - inference_start
                save_results(dataset, model_name, condition, results)
                timing = {
                    "Dataset": dataset, "Model Name": model_name, "CoT": condition,
                    "Questions": len(results), "Load seconds": round(load_seconds, 2),
                    "Inference seconds": round(inference_seconds, 2),
                    "Seconds/question": round(inference_seconds / len(results), 3),
                }
                RUN_TIMINGS.append(timing)
                with open(result_path(dataset, model_name, condition) + ".timing.json", "w") as f:
                    json.dump(timing, f, indent=2)
                print(f"Load: {load_seconds:.1f}s | Inference: {inference_seconds:.1f}s | "
                      f"Per question: {inference_seconds / len(results):.2f}s")
                print(f"Completed {len(results)} open-question predictions. Metrics follow in section 6.")
            except Exception as e:
                raise RuntimeError(
                    f"Evaluation stopped at {repo}. Completed results are saved. "
                    f"Fix this error and rerun to resume: {type(e).__name__}: {e}"
                ) from e
            finally:
                model = tokenizer = None
                gc.collect()
                torch.cuda.empty_cache()

print(f"\nSelected runs finished in {(time.time() - run_start) / 60:.1f} min")


[ACCESS OK] Arup330/BrainFace_open_CoT_Lingshu-7B_lora
[ACCESS OK] Arup330/BrainFace_open_noCoT_Lingshu-7B_lora
[ACCESS OK] Arup330/BrainFace_open_CoT_MedGemma-4B_lora
[ACCESS OK] Arup330/BrainFace_open_noCoT_MedGemma-4B_lora
[ACCESS OK] Arup330/BrainFace_open_CoT_Qwen2.5-VL-7B_lora
[ACCESS OK] Arup330/BrainFace_open_noCoT_Qwen2.5-VL-7B_lora
[ACCESS OK] Arup330/BrainFace_open_CoT_Qwen3-VL-8B_lora
[ACCESS OK] Arup330/BrainFace_open_noCoT_Qwen3-VL-8B_lora
[ACCESS OK] Arup330/BrainFace_open_CoT_Llama-11B_lora
[ACCESS OK] Arup330/BrainFace_open_noCoT_Llama-11B_lora
[ACCESS OK] Arup330/Neck_open_CoT_Lingshu-7B_lora
[ACCESS OK] Arup330/Neck_open_noCoT_Lingshu-7B_lora
[ACCESS OK] Arup330/Neck_open_CoT_MedGemma-4B_lora
[ACCESS OK] Arup330/Neck_open_noCoT_MedGemma-4B_lora
[ACCESS OK] Arup330/Neck_open_CoT_Qwen2.5-VL-7B_lora
[ACCESS OK] Arup330/Neck_open_noCoT_Qwen2.5-VL-7B_lora
[ACCESS OK] Arup330/Neck_open_CoT_Qwen3-VL-8B_lora
[ACCESS OK] Arup330/Neck_open_noCoT_Qwen3-VL-8B_lora
[ACCESS OK] Ar

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2026.10.2: Fast Qwen2_5_Vl patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

    [0/17] gold='Head' pred='head.' raw='Head.'
    [1/17] gold='Center' pred='brain stem.' raw='Brain Stem.'
    [2/17] gold='Head' pred='head.' raw='Head.'
    [3/17] gold='Head' pred='head.' raw='Head.'
    [4/17] gold='White' pred='black.' raw='Black.'
    [5/17] gold='Brain' pred='brain.' raw='Brain.'
    [6/17] gold='0' pred='2.' raw='2.'
    [7/17] gold='Control heartbeat and breathing' pred='sensory perception.' raw='Sensory perception.'
    [8/17] gold='CT' pred='ct scan.' raw='CT scan.'
    [9/17] gold='Transverse Plane' pred='transverse plane.' raw='Transverse Plane.'
    [10/17] gold='Top' pred='right jaw.' raw='Right Jaw.'
    [11/17] gold='Head' pred='head, neck' raw='Head, Neck'
    [12/17] gold='Gray' pred='gray.' raw='Gray.'
    [13/17] gold='Brain' pred='brain.' raw='Brain.'
    [14/17] gold='2' pred='2.' raw='2.'
    [15/17] gold='Cutting, chewing, maintaining facial contour and assisting pronunciation' pred='respiratory system.' raw='Respiratory system.'
    [16/17]

...

    [0/19] gold='CT' pred='computed tomography (ct)' raw='Computed Tomography (CT)'
    [1/19] gold='Transverse Plane' pred='transverse' raw='Transverse'
    [2/19] gold='Neck' pred='**answer:** neck' raw='**Answer:** Neck'
    [3/19] gold='Oval' pred='the larynx in this image is triangular.' raw='The larynx in this image is triangular.'
    [4/19] gold='Larynx' pred='lungs' raw='Lungs'
    [5/19] gold='Transverse Plane' pred='**answer:** transverse' raw='**Answer:** Transverse'
    [6/19] gold='Neck' pred='**answer:** neck' raw='**Answer:** Neck'
    [7/19] gold='Under the trachea' pred='the esophagus is located in the center of the image.' raw='The esophagus is located in the center of the image.'
    [8/19] gold='Neck' pred='abdomen' raw='Abdomen'
    [9/19] gold='Black' pred='white' raw='White'
    [10/19] gold='Irregular' pred='the trachea is u-shaped.' raw='The trachea is U-shaped.'
    [11/19] gold='Gas delivery' pred='the black hollow tissue is the trachea.' raw='The black holl

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

    [0/160] gold='CT' pred='the image is a computed tomography (ct) scan, which is a medical imaging technique that uses x-rays to produce detailed cross-sectional images of the body.' raw='The image is a computed tomography (CT) scan, which is a medical imaging technique that uses X-rays to produce detailed cross-sectional images of the body.'
    [1/160] gold='Chest' pred='the image shows a cross-sectional view of the human torso, with the rib cage and spine visible. the purpose of the image is to provide a detailed view of' raw='The image shows a cross-sectional view of the human torso, with the rib cage and spine visible. The purpose of the image is to provide a detailed view of'
    [2/160] gold='Lung' pred='the main organ in the image is the lungs.' raw='The main organ in the image is the lungs.'
    [3/160] gold='Lung' pred='the largest organ in the picture is the lung.' raw='The largest organ in the picture is the lung.'
    [4/160] gold='Lung Cancer' pred='the image shows a ct

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

    [0/160] gold='CT' pred='ct' raw='CT'
    [1/160] gold='Chest' pred='chest' raw='Chest'
    [2/160] gold='Lung' pred='lungs' raw='Lungs'
    [3/160] gold='Lung' pred='lungs' raw='Lungs'
    [4/160] gold='Lung Cancer' pred='cancer' raw='Cancer'
    [5/160] gold='Left Lung, Right' pred='left lung' raw='Left Lung'
    [6/160] gold='CT' pred='x-ray' raw='X-ray'
    [7/160] gold='Abdomen' pred='abdomen' raw='Abdomen'
    [8/160] gold='Liver' pred='liver' raw='Liver'
    [9/160] gold='Left' pred='liver' raw='Liver'
    [10/160] gold='2' pred='1' raw='1'
    [11/160] gold='Left Kidney' pred='liver' raw='Liver'
    [12/160] gold='Liver' pred='liver' raw='Liver'
    [13/160] gold='Transverse  Plane' pred='transverse plane' raw='Transverse Plane'
    [14/160] gold='CT' pred='x-ray' raw='X-ray'
    [15/160] gold='Abdomen' pred='abdomen' raw='Abdomen'
    [16/160] gold='4' pred='2' raw='2'
    [17/160] gold='Lung' pred='liver' raw='Liver'
    [18/160] gold='Top' pred='liver' raw='Liver'
    [19

## 6. BLEU, ROUGE, and BERTScore


In [12]:
# Metrics use the complete generated answer; outputs are never clipped to 4 words.
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import BERTScorer
from importlib.metadata import version as package_version

METRIC_SETTINGS = {
    "normalization": "Unicode NFKC, lowercase, collapsed whitespace; punctuation retained",
    "BLEU": "corpus BLEU-4, whitespace tokens, NLTK method1 smoothing",
    "ROUGE": "mean per-example F1, rouge-score tokenizer, no stemming",
    "BERTScore_model": BERTSCORE_MODEL,
    "BERTScore_rescale_with_baseline": False,
    "BERTScore_idf": False,
    "BERTScore_use_fast_tokenizer": False,
    "empty_prediction": "retained; BERTScore P/R/F1 set to zero",
    "versions": {p: package_version(p) for p in ("nltk", "rouge-score", "bert-score", "transformers")},
}

def compute_open_metrics(results, scorer):
    references = [normalize_answer(r["gold"]) for r in results]
    predictions = [normalize_answer(r["raw"]) for r in results]
    if not references or any(not r for r in references):
        raise ValueError("Every test question must have a nonempty reference answer.")
    bleu = corpus_bleu(
        [[reference.split()] for reference in references],
        [prediction.split() for prediction in predictions],
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=SmoothingFunction().method1,
    )
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
    rouge_rows = [rouge.score(reference, prediction)
                  for reference, prediction in zip(references, predictions)]
    # Empty responses must count against the model, not disappear from metrics.
    nonempty = [i for i, prediction in enumerate(predictions) if prediction]
    totals = {"P": 0.0, "R": 0.0, "F1": 0.0}
    if nonempty:
        precision, recall, f1 = scorer.score(
            [predictions[i] for i in nonempty],
            [references[i] for i in nonempty],
            batch_size=BERTSCORE_BATCH_SIZE,
        )
        totals = {"P": precision.sum().item(), "R": recall.sum().item(), "F1": f1.sum().item()}
    n = len(results)
    return {
        "BLEU": float(bleu),
        "ROUGE-1": sum(r["rouge1"].fmeasure for r in rouge_rows) / n,
        "ROUGE-2": sum(r["rouge2"].fmeasure for r in rouge_rows) / n,
        "ROUGE-L": sum(r["rougeL"].fmeasure for r in rouge_rows) / n,
        "BERTScore F1": totals["F1"] / n,
        "BERTScore P": totals["P"] / n,
        "BERTScore R": totals["R"] / n,
    }

required_runs = [(d, m, c) for d in DATASETS for m in MODELS for c, _ in CONDITIONS]
unfinished = [f"{d} | {m} | {c}" for d, m, c in required_runs if not is_done(d, m, c)]
if unfinished:
    raise RuntimeError("All 30 predictions are required before scoring. Remaining:\n" + "\n".join(unfinished))

# The VLM has been released before loading this much smaller metric model.
gc.collect()
torch.cuda.empty_cache()
metric_device = "cuda:0" if torch.cuda.is_available() else "cpu"
scorer = BERTScorer(
    model_type=BERTSCORE_MODEL, lang="en", device=metric_device,
    batch_size=BERTSCORE_BATCH_SIZE, idf=False,
    rescale_with_baseline=False, use_fast_tokenizer=False,
)
METRIC_SETTINGS["BERTScore_hash"] = scorer.hash
summary_rows = []
try:
    for dataset, model_name, condition in required_runs:
        path = result_path(dataset, model_name, condition)
        with open(path, encoding="utf-8") as f:
            results = json.load(f)
        metrics = compute_open_metrics(results, scorer)
        summary_rows.append({"Dataset": dataset, "Model Name": model_name,
                             "CoT": condition, **metrics})
        with open(path + ".metrics.json", "w", encoding="utf-8") as f:
            json.dump({"run_signature": run_signature(dataset, model_name, condition),
                       "settings": METRIC_SETTINGS, "metrics": metrics}, f, indent=2)
        print(f"\nOpen-question metrics -- {model_name} ({condition} CoT, {dataset}):")
        print(pd.DataFrame([metrics]).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
finally:
    scorer = None
    gc.collect()
    torch.cuda.empty_cache()

summary_df = pd.DataFrame(summary_rows)
assert len(summary_df) == 30


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Open-question metrics -- Lingshu-7B (With CoT, BrainFace):
  BLEU  ROUGE-1  ROUGE-2  ROUGE-L  BERTScore F1  BERTScore P  BERTScore R
0.0097   0.6078   0.0588   0.6078        0.9292       0.9406       0.9188

Open-question metrics -- Lingshu-7B (Without CoT, BrainFace):
  BLEU  ROUGE-1  ROUGE-2  ROUGE-L  BERTScore F1  BERTScore P  BERTScore R
0.0255   0.6471   0.0588   0.6471        0.9605       0.9623       0.9590

Open-question metrics -- MedGemma-4B (With CoT, BrainFace):
  BLEU  ROUGE-1  ROUGE-2  ROUGE-L  BERTScore F1  BERTScore P  BERTScore R
0.0036   0.6103   0.0000   0.6103        0.9342       0.9317       0.9368

Open-question metrics -- MedGemma-4B (Without CoT, BrainFace):
  BLEU  ROUGE-1  ROUGE-2  ROUGE-L  BERTScore F1  BERTScore P  BERTScore R
0.0131   0.6686   0.0000   0.6686        0.9472       0.9479       0.9466

Open-question metrics -- Qwen2.5-VL-7B (With CoT, BrainFace):
  BLEU  ROUGE-1  ROUGE-2  ROUGE-L  BERTScore F1  BERTScore P  BERTScore R
0.0121   0.4314   0.000

## 7. Final summary tables


In [13]:
# 7. Final open-ended summary: all models and both training variants.
# Rebuild from validated saved metrics, rather than trusting an old summary_df
# left in notebook memory after changes to the prompt, data, or model settings.
required_runs = [(d, m, c) for d in DATASETS for m in MODELS for c, _ in CONDITIONS]
metric_columns = ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "BERTScore F1", "BERTScore P", "BERTScore R"]
if METRIC_SETTINGS["BERTScore_model"] != BERTSCORE_MODEL:
    raise RuntimeError("BERTScore model setting changed. Rerun section 6.")
validated_rows = []
for dataset, model_name, condition in required_runs:
    if not is_done(dataset, model_name, condition):
        raise RuntimeError(f"Current predictions required: {dataset} / {model_name} / {condition}. Rerun section 5.")
    path = result_path(dataset, model_name, condition) + ".metrics.json"
    try:
        with open(path, encoding="utf-8") as f:
            saved = json.load(f)
    except (OSError, ValueError) as e:
        raise RuntimeError(f"Metrics unavailable for {dataset} / {model_name} / {condition}. Rerun section 6.") from e
    if saved.get("run_signature") != run_signature(dataset, model_name, condition) or saved.get("settings") != METRIC_SETTINGS:
        raise RuntimeError(f"Stale metrics for {dataset} / {model_name} / {condition}. Rerun section 6.")
    validated_rows.append({"Dataset": dataset, "Model Name": model_name, "CoT": condition,
                           **{key: saved["metrics"][key] for key in metric_columns}})
summary_df = pd.DataFrame(validated_rows)
if len(summary_df) != 30:
    raise RuntimeError("All 30 metric rows are required for the final summary.")
scope = "FULL TEST SET" if MAX_ROWS_PER_DATASET is None else f"SMOKE TEST: first {MAX_ROWS_PER_DATASET} rows per dataset"
columns = ["Dataset", "Model Name", "CoT", "BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L",
           "BERTScore F1", "BERTScore P", "BERTScore R"]
summary_df = summary_df[columns]
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 240)
report = [f"OPEN-ENDED EVALUATION — {scope} — 30/30 MODELS", 
          "Inputs: image + identical question prompt for both training variants",
          "Scores are decimals. ROUGE columns are mean F1. BLEU is corpus BLEU-4."]
for dataset in DATASETS:
    part = summary_df[summary_df["Dataset"] == dataset]
    assert len(part) == 10
    heading = f"OPEN-ENDED SUMMARY TABLE — {dataset} — BLEU/ROUGE/BERTScore, all models, both conditions"
    table = part.to_string(index=False, float_format=lambda v: f"{v:.4f}")
    print("\n" + "=" * 170 + "\n" + heading + "\n" + scope + "\n" + "=" * 170)
    print(table)
    part.to_csv(os.path.join(RESULTS_DIR, f"open_comparison_summary_{dataset}.csv"), index=False, float_format="%.6f")
    raw = {}
    for model_name in MODELS:
        for condition, _ in CONDITIONS:
            with open(result_path(dataset, model_name, condition), encoding="utf-8") as f:
                raw[f"{model_name}_{condition}"] = json.load(f)
    with open(os.path.join(RESULTS_DIR, f"open_comparison_raw_results_{dataset}.json"), "w", encoding="utf-8") as f:
        json.dump(raw, f, indent=2, ensure_ascii=False)
    report.extend(["\n" + heading, table])
print("\nALL DATASETS — 30 MODELS — " + scope)
print(summary_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
summary_df.to_csv(os.path.join(RESULTS_DIR, "open_comparison_summary_ALL.csv"), index=False, float_format="%.6f")
report.extend(["\nALL DATASETS — 30 MODELS", summary_df.to_string(index=False, float_format=lambda v: f"{v:.4f}")])
diagnostics = []
for dataset, model_name, condition in required_runs:
    with open(result_path(dataset, model_name, condition), encoding="utf-8") as f:
        rows = json.load(f)
    diagnostics.append({"Dataset": dataset, "Model Name": model_name, "CoT": condition,
                        "N": len(rows), "Empty": sum(not normalize_answer(r["raw"]) for r in rows),
                        "Over 4 words": sum(len(r["raw"].split()) > 4 for r in rows)})
diagnostics_df = pd.DataFrame(diagnostics)
diagnostics_df.to_csv(os.path.join(RESULTS_DIR, "open_output_diagnostics.csv"), index=False)
report.extend(["\nOUTPUT DIAGNOSTICS (all outputs retained)", diagnostics_df.to_string(index=False),
               "\nMETRIC SETTINGS", json.dumps(METRIC_SETTINGS, indent=2)])
with open(os.path.join(RESULTS_DIR, "open_metric_settings.json"), "w", encoding="utf-8") as f:
    json.dump({"scope": scope, "prompt": systemPrompt("{question}"), **METRIC_SETTINGS}, f, indent=2)
report_path = os.path.join(RESULTS_DIR, "FINAL_OPEN_MODEL_SUMMARY.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report))
print(f"\nSaved full tables, raw answers, metric settings, and complete report in {RESULTS_DIR}")
print(f"Complete report: {report_path}")



OPEN-ENDED SUMMARY TABLE — BrainFace — BLEU/ROUGE/BERTScore, all models, both conditions
FULL TEST SET
  Dataset    Model Name     CoT   BLEU  ROUGE-1  ROUGE-2  ROUGE-L  BERTScore F1  BERTScore P  BERTScore R
BrainFace    Lingshu-7B    With 0.0097   0.6078   0.0588   0.6078        0.9292       0.9406       0.9188
BrainFace    Lingshu-7B Without 0.0255   0.6471   0.0588   0.6471        0.9605       0.9623       0.9590
BrainFace   MedGemma-4B    With 0.0036   0.6103   0.0000   0.6103        0.9342       0.9317       0.9368
BrainFace   MedGemma-4B Without 0.0131   0.6686   0.0000   0.6686        0.9472       0.9479       0.9466
BrainFace Qwen2.5-VL-7B    With 0.0121   0.4314   0.0000   0.4314        0.9134       0.9123       0.9147
BrainFace Qwen2.5-VL-7B Without 0.0150   0.4804   0.0000   0.4804        0.9152       0.9110       0.9200
BrainFace   Qwen3-VL-8B    With 0.0094   0.2108   0.0000   0.2108        0.8735       0.8536       0.8953
BrainFace   Qwen3-VL-8B Without 0.0116   0.2686 